# Polarization

The polarization of the CMB is described by the Stokes parameters $Q$ and $U$. Unlike the
temperature $T$ (or intensity $I$), they are not scalars: they depend on the orientation of
the local reference frame, and the combinations $Q \pm iU$ transform as spin-2 fields on the
sphere. Their spin-weighted harmonic transform splits them into two scalar components,
the curl-free **E modes** and the divergence-free **B modes**:

$$
Q \pm iU = -\sum_{\ell m} \left(a^E_{\ell m} \pm i\, a^B_{\ell m}\right) {}_{\pm 2}Y_{\ell m}.
$$

jax-healpy follows the healpy conventions for these transforms. This page covers:

- simulating polarized maps from power spectra with `synfast`;
- the polarized transforms between $T, Q, U$ maps and $T, E, B$ alms;
- the spin transforms `map2alm_spin` and `alm2map_spin`;
- the six $TT, EE, BB, TE, EB, TB$ power spectra with `anafast`;
- smoothing, just-in-time compilation and differentiation of these transforms.

!!! note
    The spherical harmonic transforms require the `recommended` extra (s2fft), see the
    [installation](installation.md) page.

In [ ]:
import jax

# Enable 64-bit precision before importing jax_healpy, which warns otherwise.
jax.config.update('jax_enable_x64', True)

import jax.numpy as jnp
import numpy as np

import jax_healpy as hp

## Simulating polarized maps

We use a toy model for the power spectra, with $C_\ell \propto 1/(\ell + 10)^2$, an
$EE$ spectrum 20 times smaller than $TT$, a $BB$ spectrum 10 times smaller than $EE$, and a
correlation between $T$ and $E$. The polarization spectra vanish at $\ell < 2$, since a
spin-2 field has no monopole or dipole.

With `lmax = 3 * nside - 1`, the default of the transforms, the spectra are arrays of
`lmax + 1` values.

In [ ]:
nside = 32
lmax = 3 * nside - 1
ell = np.arange(lmax + 1)

shape = np.where(ell >= 2, 1 / (ell + 10.0) ** 2, 0.0)
cl_tt = 1000 * shape
cl_ee = 50 * shape
cl_bb = 5 * shape
cl_te = 0.5 * np.sqrt(cl_tt * cl_ee)

Given a list of four spectra, [`synfast`][jax_healpy.synfast] draws correlated $T, E, B$
alms and returns the $T, Q, U$ maps, stacked along the first axis. As in healpy,
`new=True` means that the spectra are ordered as $TT, EE, BB, TE$. Like the other random
functions of JAX, `synfast` takes a PRNG key as first argument.

In [ ]:
key = jax.random.key(0)
maps, alms = hp.synfast(key, [cl_tt, cl_ee, cl_bb, cl_te], nside, lmax=lmax, new=True, alm=True)

print(f'Maps: {maps.shape}')  # (T, Q, U) x npix
print(f'Alms: {alms.shape}')  # (T, E, B) x (lmax + 1) x (2 * lmax + 1)

The alms use the s2fft layout by default: a two-dimensional array indexed by $\ell$ and by
$m + \ell_\mathrm{max}$, which also stores the negative $m$. Pass `healpy_ordering=True` to
the transforms to work with the one-dimensional healpy layout instead.

## From maps to alms

Given a stack of three maps, [`map2alm`][jax_healpy.map2alm] interprets them as $I, Q, U$
and returns the $T, E, B$ alms.

In [ ]:
alms_teb = hp.map2alm(maps, lmax=lmax)
print(f'Alms: {alms_teb.shape}')

As in healpy, this transform is only approximate: the HEALPix grid has no exact quadrature
rule, and the `iter` parameter (3 by default) sets the number of iterations that refine
the alms. Above $\ell \approx 2\,n_\mathrm{side}$, the recovered alms differ from the input
ones by a few percent.

Unlike healpy, jax-healpy also accepts a stack of two maps, interpreted as $Q, U$ only, and
then returns the $E, B$ alms. This avoids transforming a temperature map when only the
polarization is needed.

In [ ]:
alms_eb = hp.map2alm(maps[1:], lmax=lmax)
print(f'Alms: {alms_eb.shape}')
print(f'Same E, B alms as from T, Q, U maps: {jnp.allclose(alms_eb, alms_teb[1:])}')

## Spin transforms

[`map2alm_spin`][jax_healpy.map2alm_spin] and [`alm2map_spin`][jax_healpy.alm2map_spin]
are the spin-weighted transforms on which the polarized transforms are built. They take a
list of two maps or alms and the spin, 2 for polarization. With `spin=2`, `map2alm_spin`
gives the same $E, B$ alms as `map2alm`. Note that `map2alm_spin` does not iterate by
default (`iter=0`), as in healpy.

In [ ]:
alm_e, alm_b = hp.map2alm_spin([maps[1], maps[2]], spin=2, lmax=lmax)
alms_eb_no_iter = hp.map2alm(maps[1:], lmax=lmax, iter=0)
print(f'Same alms as map2alm(iter=0): {jnp.allclose(jnp.stack([alm_e, alm_b]), alms_eb_no_iter)}')

The spin transforms also handle other spins, for example spin 1 for the gradient of a
scalar field, or spin 0, in which case they act on a single map and are equivalent to
`map2alm` and `alm2map` for a temperature map.

## From alms to maps

[`alm2map`][jax_healpy.alm2map] is the inverse of `map2alm`: it turns a stack of $T, E, B$
alms into $T, Q, U$ maps, or a stack of $E, B$ alms into $Q, U$ maps.

To see what E and B modes look like, we keep only the E modes, then only the B modes, of
the simulated alms. A pure E-mode map has no B-mode power, and vice versa.

In [ ]:
alm_zero = jnp.zeros_like(alms[1])
q_e, u_e = hp.alm2map(jnp.stack([alms[1], alm_zero]), nside, lmax=lmax)
q_b, u_b = hp.alm2map_spin([alm_zero, alms[2]], nside, spin=2, lmax=lmax)

cl_e_only = hp.anafast(jnp.stack([q_e, u_e]), lmax=lmax)
cl_b_only = hp.anafast(jnp.stack([q_b, u_b]), lmax=lmax)
print(f'E-mode maps: EE power = {cl_e_only[0].sum():.2e}, BB power = {cl_e_only[1].sum():.2e}')
print(f'B-mode maps: EE power = {cl_b_only[0].sum():.2e}, BB power = {cl_b_only[1].sum():.2e}')

The leakage between E and B is small but not zero, since `anafast` relies on the
approximate `map2alm` transform.

## Power spectra

For $I, Q, U$ maps, [`anafast`][jax_healpy.anafast] returns the six auto- and
cross-spectra, in the healpy order $TT, EE, BB, TE, EB, TB$. For $Q, U$ maps, it returns
the three spectra $EE, BB, EB$.

We compare them with the input spectra, averaged over bands of $\ell$ to reduce the cosmic
variance of a single realization.

In [ ]:
cls = hp.anafast(maps, lmax=lmax)
print(f'Spectra: {cls.shape}')

inputs = [cl_tt, cl_ee, cl_bb, cl_te, 0 * shape, 0 * shape]
bands = [(2, 20), (20, 50), (50, lmax + 1)]
print('       ' + ''.join(f'  ell {lo:2}-{hi - 1:2}  ' for lo, hi in bands))
for name, cl_out, cl_in in zip(['TT', 'EE', 'BB', 'TE', 'EB', 'TB'], cls, inputs):
    row = ''.join(f'{cl_out[lo:hi].mean():9.2e} ({cl_in[lo:hi].mean():8.1e})' for lo, hi in bands)
    print(f'{name}: {row}')

The measured spectra (with the input ones in parentheses) agree within the cosmic
variance, which is larger at low $\ell$, where fewer modes are available. The cross-spectra
$EB$ and $TB$ vanish in this model and are only noise.

## Smoothing

`synfast`, `alm2map` and [`smoothing`][jax_healpy.smoothing] accept a Gaussian beam,
specified by its full width at half maximum `fwhm` in radians. For polarization, the beam
window of the E and B modes includes the spin-2 correction. Here, a 3° beam reduces the
amplitude of the $Q$ map:

In [ ]:
fwhm = float(np.radians(3))
maps_smoothed = hp.alm2map(alms, nside, lmax=lmax, fwhm=fwhm)

print(f'Standard deviation of Q: {maps[1].std():.3f} -> {maps_smoothed[1].std():.3f} after smoothing')

## JIT compilation and differentiation

The polarized transforms are JAX functions: they can be compiled with `jax.jit` and
differentiated with `jax.grad`. The parameters that set the shape of the arrays, such as
`nside`, `lmax` or the spin, must be static.

As an example, we compute the B-mode power of a pair of $Q, U$ maps and its gradient with
respect to the maps. The gradient points in the direction in which the maps should be
changed to reduce their B-mode content, the basis of E/B separation by optimization.

In [ ]:
@jax.jit
def b_mode_power(qu):
    _, alm_b = hp.map2alm_spin([qu[0], qu[1]], spin=2, lmax=lmax)
    return jnp.sum(jnp.abs(alm_b) ** 2)


qu = maps[1:]
power, gradient = jax.value_and_grad(b_mode_power)(qu)
print(f'B-mode power: {power:.3e}')
print(f'Gradient: {gradient.shape}')

# One step of gradient descent reduces the B-mode power.
step = 0.1 * power / jnp.sum(gradient**2)
print(f'B-mode power after one step: {b_mode_power(qu - step * gradient):.3e}')